In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install required libraries
!pip install PyPDF2 spacy scikit-learn flask pyngrok
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 84.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import PyPDF2
import spacy
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load('en_core_web_sm')

print("✅ All libraries imported!")
print("✅ NLP model loaded!")

✅ All libraries imported!
✅ NLP model loaded!


In [ ]:
def extract_text_from_pdf(pdf_path):
    """
    Extracts all text from a PDF file
    Works with any resume PDF
    """
    text = ""

    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)

        # Loop through every page
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text += page.extract_text()

    return text

# Test it with a sample text for now
sample_resume_text = """
John Doe
Email: john@email.com | Phone: 9876543210

SKILLS
Python, Machine Learning, Deep Learning, SQL, TensorFlow,
Data Analysis, Pandas, NumPy, Matplotlib, Flask, Git, GitHub

EDUCATION
B.Tech Computer Science - 2026

EXPERIENCE
Data Science Intern - CodTech IT Solutions
- Built ETL pipeline using Pandas and Scikit-learn
- Developed NLP model using TensorFlow achieving 88% accuracy
- Deployed ML model using Flask REST API

PROJECTS
IPL Data Pipeline - ETL pipeline on cricket dataset
Resume Screener - AI tool to match resumes with job descriptions
"""

print("✅ PDF extractor function ready!")
print(f"   Sample resume length: {len(sample_resume_text)} characters")

✅ PDF extractor function ready!
   Sample resume length: 557 characters


In [ ]:
def preprocess_text(text):
    """
    Cleans and normalizes text for comparison
    Removes noise, keeps important words
    """
    # Convert to lowercase
    text = text.lower()

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Use spaCy to remove stopwords and lemmatize
    # Lemmatize = convert words to base form
    # e.g. "running" → "run", "models" → "model"
    doc = nlp(text)

    cleaned_words = [
        token.lemma_ for token in doc
        if not token.is_stop        # Remove stopwords (the, is, at...)
        and not token.is_punct      # Remove punctuation
        and len(token.text) > 2     # Remove very short words
    ]

    return ' '.join(cleaned_words)

# Test it
sample_cleaned = preprocess_text(sample_resume_text)
print("✅ Text preprocessor ready!")
print()
print("Before cleaning:")
print(sample_resume_text[:200])
print()
print("After cleaning:")
print(sample_cleaned[:200])

✅ Text preprocessor ready!

Before cleaning:

John Doe
Email: john@email.com | Phone: 9876543210

SKILLS
Python, Machine Learning, Deep Learning, SQL, TensorFlow,
Data Analysis, Pandas, NumPy, Matplotlib, Flask, Git, GitHub

EDUCATION
B.Tech Com

After cleaning:
john doe email john email com phone skill python machine learn deep learning sql tensorflow datum analysis panda numpy matplotlib flask git github education tech computer science experience datum scie


In [ ]:
# Comprehensive list of Data Science & Tech skills
SKILLS_DATABASE = [
    # Programming
    'python', 'java', 'javascript', 'sql', 'r', 'scala', 'c++',
    # ML/AI
    'machine learning', 'deep learning', 'neural network', 'nlp',
    'computer vision', 'tensorflow', 'pytorch', 'keras', 'scikit-learn',
    # Data
    'pandas', 'numpy', 'matplotlib', 'seaborn', 'tableau', 'power bi',
    'data analysis', 'data visualization', 'statistics',
    # Engineering
    'flask', 'fastapi', 'django', 'docker', 'kubernetes', 'aws',
    'azure', 'gcp', 'git', 'github', 'rest api',
    # Database
    'mysql', 'postgresql', 'mongodb', 'sqlite', 'redis',
    # Other
    'excel', 'spark', 'hadoop', 'airflow', 'etl pipeline',
    'random forest', 'xgboost', 'linear regression', 'classification'
]

def extract_skills(text):
    """
    Finds which skills from our database
    are present in the given text
    """
    text_lower = text.lower()
    found_skills = []

    for skill in SKILLS_DATABASE:
        if skill in text_lower:
            found_skills.append(skill)

    return found_skills

# Test it
resume_skills = extract_skills(sample_resume_text)
print("✅ Skills extractor ready!")
print(f"   Skills found in sample resume: {len(resume_skills)}")
print()
print("Skills detected:")
for skill in resume_skills:
    print(f"   ✅ {skill}")

✅ Skills extractor ready!
   Skills found in sample resume: 17

Skills detected:
   ✅ python
   ✅ sql
   ✅ r
   ✅ machine learning
   ✅ deep learning
   ✅ nlp
   ✅ tensorflow
   ✅ scikit-learn
   ✅ pandas
   ✅ numpy
   ✅ matplotlib
   ✅ data analysis
   ✅ flask
   ✅ git
   ✅ github
   ✅ rest api
   ✅ etl pipeline


In [ ]:
def calculate_match_score(resume_text, job_description):
    """
    Calculates how well a resume matches a job description
    Returns a score from 0-100%
    Uses TF-IDF + Cosine Similarity
    """
    # Clean both texts
    cleaned_resume = preprocess_text(resume_text)
    cleaned_job    = preprocess_text(job_description)

    # Convert texts to numerical vectors using TF-IDF
    # TF-IDF = Term Frequency-Inverse Document Frequency
    # Words that appear often in resume but are unique = more important
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([cleaned_resume, cleaned_job])

    # Calculate cosine similarity between the two vectors
    # 1.0 = identical, 0.0 = completely different
    similarity = cosine_similarity(vectors[0], vectors[1])[0][0]

    # Convert to percentage
    score = round(similarity * 100, 2)

    return score

def analyze_gap(resume_text, job_description):
    """
    Finds skills in job description that are
    missing from the resume
    """
    resume_skills = extract_skills(resume_text)
    job_skills    = extract_skills(job_description)

    # Skills in job but NOT in resume = gap
    missing_skills = [s for s in job_skills if s not in resume_skills]
    matching_skills = [s for s in job_skills if s in resume_skills]

    return matching_skills, missing_skills

print("✅ Match engine ready!")
print("✅ Gap analyzer ready!")

✅ Match engine ready!
✅ Gap analyzer ready!


In [ ]:
# Sample job description - Data Science role
sample_job = """
We are looking for a Data Scientist with the following skills:

REQUIREMENTS:
- Strong programming skills in Python and SQL
- Experience with Machine Learning and Deep Learning
- Knowledge of TensorFlow or PyTorch
- Experience with Flask or FastAPI for model deployment
- Familiarity with Docker and AWS
- Experience with data visualization using Tableau or Power BI
- Knowledge of Git and GitHub
- Strong understanding of Statistics and Data Analysis

NICE TO HAVE:
- Experience with Spark or Hadoop
- Knowledge of MongoDB or PostgreSQL
- XGBoost or Random Forest experience
"""

# Run the complete analysis
match_score = calculate_match_score(sample_resume_text, sample_job)
matching_skills, missing_skills = analyze_gap(sample_resume_text, sample_job)

# Display results
print("=" * 50)
print("      📊 RESUME MATCH REPORT")
print("=" * 50)
print(f"\n  Overall Match Score : {match_score}%")
print()

# Score interpretation
if match_score >= 75:
    print("  Status : 🟢 STRONG MATCH — Apply confidently!")
elif match_score >= 50:
    print("  Status : 🟡 MODERATE MATCH — Some gaps to fill")
else:
    print("  Status : 🔴 WEAK MATCH — Significant gaps found")

print()
print(f"  ✅ Matching Skills ({len(matching_skills)}):")
for skill in matching_skills:
    print(f"     • {skill}")

print()
print(f"  ❌ Missing Skills ({len(missing_skills)}):")
for skill in missing_skills:
    print(f"     • {skill}")

print()
print("  💡 Recommendations:")
for skill in missing_skills[:3]:
    print(f"     → Learn {skill} to improve your match score")

print("=" * 50)

      📊 RESUME MATCH REPORT

  Overall Match Score : 19.97%

  Status : 🔴 WEAK MATCH — Significant gaps found

  ✅ Matching Skills (10):
     • python
     • sql
     • r
     • machine learning
     • deep learning
     • tensorflow
     • data analysis
     • flask
     • git
     • github

  ❌ Missing Skills (14):
     • pytorch
     • tableau
     • power bi
     • data visualization
     • statistics
     • fastapi
     • docker
     • aws
     • postgresql
     • mongodb
     • spark
     • hadoop
     • random forest
     • xgboost

  💡 Recommendations:
     → Learn pytorch to improve your match score
     → Learn tableau to improve your match score
     → Learn power bi to improve your match score


In [ ]:
def calculate_final_score(resume_text, job_description):
    """
    Improved scoring that combines:
    1. TF-IDF cosine similarity (40% weight)
    2. Skills match percentage (60% weight)
    This gives a more meaningful and accurate score!
    """
    # Score 1: TF-IDF similarity
    tfidf_score = calculate_match_score(resume_text, job_description)

    # Score 2: Skills match percentage
    resume_skills   = extract_skills(resume_text)
    job_skills      = extract_skills(job_description)

    if len(job_skills) == 0:
        skills_score = 0
    else:
        matched = len([s for s in job_skills if s in resume_skills])
        skills_score = (matched / len(job_skills)) * 100

    # Weighted final score
    # Skills match matters more than word similarity
    final_score = (tfidf_score * 0.4) + (skills_score * 0.6)
    final_score = round(final_score, 2)

    return final_score, tfidf_score, skills_score

# Test improved score
final, tfidf, skills = calculate_final_score(sample_resume_text, sample_job)

print("=" * 50)
print("      📊 IMPROVED MATCH REPORT")
print("=" * 50)
print(f"\n  TF-IDF Similarity   : {tfidf}%")
print(f"  Skills Match        : {round(skills, 2)}%")
print(f"  Final Match Score   : {final}%  ⭐")
print()

if final >= 75:
    print("  Status : 🟢 STRONG MATCH — Apply confidently!")
elif final >= 50:
    print("  Status : 🟡 MODERATE MATCH — Some gaps to fill")
else:
    print("  Status : 🔴 WEAK MATCH — Significant gaps found")

print("=" * 50)

      📊 IMPROVED MATCH REPORT

  TF-IDF Similarity   : 19.97%
  Skills Match        : 41.67%
  Final Match Score   : 32.99%  ⭐

  Status : 🔴 WEAK MATCH — Significant gaps found


In [ ]:
def screen_resume(resume_text, job_description):
    """
    Complete resume screening pipeline
    Input  : resume text + job description
    Output : full analysis report
    """
    # Calculate scores
    final_score, tfidf_score, skills_score = calculate_final_score(
                                              resume_text, job_description)

    # Get matching and missing skills
    matching_skills, missing_skills = analyze_gap(resume_text, job_description)

    # Determine status
    if final_score >= 75:
        status = "🟢 STRONG MATCH"
        advice = "Your resume is well aligned! Apply confidently."
    elif final_score >= 50:
        status = "🟡 MODERATE MATCH"
        advice = "Good foundation! Add missing skills to strengthen your profile."
    elif final_score >= 30:
        status = "🟠 PARTIAL MATCH"
        advice = "You have some relevant skills. Focus on the missing ones."
    else:
        status = "🔴 WEAK MATCH"
        advice = "Significant gaps found. Consider upskilling before applying."

    # Build report
    report = {
        'final_score'     : final_score,
        'tfidf_score'     : tfidf_score,
        'skills_score'    : skills_score,
        'status'          : status,
        'advice'          : advice,
        'matching_skills' : matching_skills,
        'missing_skills'  : missing_skills,
        'total_job_skills': len(matching_skills) + len(missing_skills),
        'matched_count'   : len(matching_skills),
        'missing_count'   : len(missing_skills)
    }

    return report

# Test the complete pipeline
report = screen_resume(sample_resume_text, sample_job)

print("=" * 55)
print("        🤖 AI RESUME SCREENER — FULL REPORT")
print("=" * 55)
print(f"  Final Score     : {report['final_score']}%")
print(f"  Status          : {report['status']}")
print(f"  Advice          : {report['advice']}")
print()
print(f"  Skills Matched  : {report['matched_count']} / {report['total_job_skills']}")
print(f"  Skills Missing  : {report['missing_count']}")
print()
print("  ✅ Your Matching Skills:")
for s in report['matching_skills']:
    print(f"     • {s}")
print()
print("  ❌ Skills to Add:")
for s in report['missing_skills']:
    print(f"     • {s}")
print()
print("  💡 Top 3 Priority Skills to Learn:")
for s in report['missing_skills'][:3]:
    print(f"     🎯 {s}")
print("=" * 55)

        🤖 AI RESUME SCREENER — FULL REPORT
  Final Score     : 32.99%
  Status          : 🟠 PARTIAL MATCH
  Advice          : You have some relevant skills. Focus on the missing ones.

  Skills Matched  : 10 / 24
  Skills Missing  : 14

  ✅ Your Matching Skills:
     • python
     • sql
     • r
     • machine learning
     • deep learning
     • tensorflow
     • data analysis
     • flask
     • git
     • github

  ❌ Skills to Add:
     • pytorch
     • tableau
     • power bi
     • data visualization
     • statistics
     • fastapi
     • docker
     • aws
     • postgresql
     • mongodb
     • spark
     • hadoop
     • random forest
     • xgboost

  💡 Top 3 Priority Skills to Learn:
     🎯 pytorch
     🎯 tableau
     🎯 power bi


In [ ]:
import os

# Create folder structure on Google Drive
base_path = '/content/drive/MyDrive/CodTech_Internship/resume_screener'

os.makedirs(f'{base_path}/templates', exist_ok=True)
os.makedirs(f'{base_path}/uploads',   exist_ok=True)

print("✅ Folder structure created!")
print(f"   {base_path}/")
print(f"   {base_path}/templates/")
print(f"   {base_path}/uploads/")

✅ Folder structure created!
   /content/drive/MyDrive/CodTech_Internship/resume_screener/
   /content/drive/MyDrive/CodTech_Internship/resume_screener/templates/
   /content/drive/MyDrive/CodTech_Internship/resume_screener/uploads/


In [ ]:
app_code = '''
import os
import PyPDF2
import re
import spacy
from flask import Flask, request, render_template, jsonify
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from werkzeug.utils import secure_filename

app = Flask(__name__)
app.config["UPLOAD_FOLDER"] = "uploads"
app.config["MAX_CONTENT_LENGTH"] = 16 * 1024 * 1024  # 16MB max

nlp = spacy.load("en_core_web_sm")

SKILLS_DATABASE = [
    "python", "java", "javascript", "sql", "r", "scala", "c++",
    "machine learning", "deep learning", "neural network", "nlp",
    "computer vision", "tensorflow", "pytorch", "keras", "scikit-learn",
    "pandas", "numpy", "matplotlib", "seaborn", "tableau", "power bi",
    "data analysis", "data visualization", "statistics",
    "flask", "fastapi", "django", "docker", "kubernetes", "aws",
    "azure", "gcp", "git", "github", "rest api",
    "mysql", "postgresql", "mongodb", "sqlite", "redis",
    "excel", "spark", "hadoop", "airflow", "etl pipeline",
    "random forest", "xgboost", "linear regression", "classification"
]

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text()
    return text

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    doc = nlp(text)
    words = [t.lemma_ for t in doc
             if not t.is_stop and not t.is_punct and len(t.text) > 2]
    return " ".join(words)

def extract_skills(text):
    text_lower = text.lower()
    return [s for s in SKILLS_DATABASE if s in text_lower]

def screen_resume(resume_text, job_description):
    # TF-IDF similarity
    cleaned_resume = preprocess_text(resume_text)
    cleaned_job    = preprocess_text(job_description)
    vectorizer     = TfidfVectorizer()
    vectors        = vectorizer.fit_transform([cleaned_resume, cleaned_job])
    tfidf_score    = round(cosine_similarity(vectors[0], vectors[1])[0][0] * 100, 2)

    # Skills match
    resume_skills   = extract_skills(resume_text)
    job_skills      = extract_skills(job_description)
    matching        = [s for s in job_skills if s in resume_skills]
    missing         = [s for s in job_skills if s not in resume_skills]
    skills_score    = round((len(matching) / len(job_skills) * 100)
                             if job_skills else 0, 2)

    # Final weighted score
    final_score = round((tfidf_score * 0.4) + (skills_score * 0.6), 2)

    # Status
    if final_score >= 75:
        status = "STRONG MATCH"
        color  = "#2ecc71"
    elif final_score >= 50:
        status = "MODERATE MATCH"
        color  = "#f39c12"
    elif final_score >= 30:
        status = "PARTIAL MATCH"
        color  = "#e67e22"
    else:
        status = "WEAK MATCH"
        color  = "#e74c3c"

    return {
        "final_score"     : final_score,
        "tfidf_score"     : tfidf_score,
        "skills_score"    : skills_score,
        "status"          : status,
        "color"           : color,
        "matching_skills" : matching,
        "missing_skills"  : missing,
        "top_3_missing"   : missing[:3],
        "matched_count"   : len(matching),
        "missing_count"   : len(missing),
        "total_skills"    : len(job_skills)
    }

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/analyze", methods=["POST"])
def analyze():
    try:
        # Get job description from form
        job_description = request.form.get("job_description", "")

        # Get resume — either PDF or text
        resume_text = ""
        if "resume_pdf" in request.files:
            file = request.files["resume_pdf"]
            if file.filename != "" and file.filename.endswith(".pdf"):
                filename = secure_filename(file.filename)
                filepath = os.path.join(app.config["UPLOAD_FOLDER"], filename)
                file.save(filepath)
                resume_text = extract_text_from_pdf(filepath)
                os.remove(filepath)  # Delete after reading

        # Fallback to text input
        if not resume_text:
            resume_text = request.form.get("resume_text", "")

        if not resume_text or not job_description:
            return jsonify({"error": "Please provide both resume and job description"})

        # Run analysis
        result = screen_resume(resume_text, job_description)
        return jsonify(result)

    except Exception as e:
        return jsonify({"error": str(e)})

if __name__ == "__main__":
    os.makedirs("uploads", exist_ok=True)
    app.run(debug=True)
'''

# Save app.py to Google Drive
with open('/content/drive/MyDrive/CodTech_Internship/resume_screener/app.py', 'w') as f:
    f.write(app_code)

print("✅ app.py saved to Google Drive!")

✅ app.py saved to Google Drive!


In [ ]:
html_code = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AI Resume Screener</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }

        body {
            font-family: 'Segoe UI', sans-serif;
            background: linear-gradient(135deg, #1a1a2e, #16213e, #0f3460);
            min-height: 100vh;
            color: #fff;
            padding: 30px 20px;
        }

        .container { max-width: 900px; margin: 0 auto; }

        .header {
            text-align: center;
            margin-bottom: 40px;
        }
        .header h1 {
            font-size: 2.5em;
            background: linear-gradient(90deg, #00d2ff, #3a7bd5);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            margin-bottom: 10px;
        }
        .header p { color: #aaa; font-size: 1.1em; }

        .card {
            background: rgba(255,255,255,0.05);
            border: 1px solid rgba(255,255,255,0.1);
            border-radius: 16px;
            padding: 30px;
            margin-bottom: 24px;
            backdrop-filter: blur(10px);
        }

        .card h2 {
            font-size: 1.2em;
            color: #00d2ff;
            margin-bottom: 16px;
        }

        textarea {
            width: 100%;
            background: rgba(255,255,255,0.08);
            border: 1px solid rgba(255,255,255,0.15);
            border-radius: 10px;
            padding: 14px;
            color: #fff;
            font-size: 0.95em;
            resize: vertical;
            outline: none;
            transition: border 0.3s;
        }
        textarea:focus { border-color: #00d2ff; }

        .file-upload {
            border: 2px dashed rgba(255,255,255,0.2);
            border-radius: 10px;
            padding: 30px;
            text-align: center;
            cursor: pointer;
            transition: all 0.3s;
            margin-bottom: 16px;
        }
        .file-upload:hover { border-color: #00d2ff; background: rgba(0,210,255,0.05); }
        .file-upload input { display: none; }
        .file-upload p { color: #aaa; }
        .file-upload span { color: #00d2ff; font-weight: bold; }

        .or-divider {
            text-align: center;
            color: #aaa;
            margin: 16px 0;
            position: relative;
        }
        .or-divider::before, .or-divider::after {
            content: "";
            position: absolute;
            top: 50%;
            width: 45%;
            height: 1px;
            background: rgba(255,255,255,0.1);
        }
        .or-divider::before { left: 0; }
        .or-divider::after { right: 0; }

        .btn {
            width: 100%;
            padding: 16px;
            background: linear-gradient(90deg, #00d2ff, #3a7bd5);
            border: none;
            border-radius: 10px;
            color: #fff;
            font-size: 1.1em;
            font-weight: bold;
            cursor: pointer;
            transition: opacity 0.3s;
            margin-top: 10px;
        }
        .btn:hover { opacity: 0.9; }
        .btn:disabled { opacity: 0.5; cursor: not-allowed; }

        /* Results */
        #results { display: none; }

        .score-circle {
            width: 150px;
            height: 150px;
            border-radius: 50%;
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            margin: 0 auto 20px;
            border: 6px solid;
        }
        .score-circle .score-num {
            font-size: 2.2em;
            font-weight: bold;
        }
        .score-circle .score-label {
            font-size: 0.75em;
            color: #aaa;
        }

        .status-badge {
            text-align: center;
            font-size: 1.3em;
            font-weight: bold;
            margin-bottom: 10px;
        }

        .advice {
            text-align: center;
            color: #aaa;
            margin-bottom: 20px;
            font-style: italic;
        }

        .score-breakdown {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 16px;
            margin-bottom: 20px;
        }
        .score-box {
            background: rgba(255,255,255,0.05);
            border-radius: 10px;
            padding: 16px;
            text-align: center;
        }
        .score-box .val {
            font-size: 1.8em;
            font-weight: bold;
            color: #00d2ff;
        }
        .score-box .lbl { color: #aaa; font-size: 0.85em; }

        .skills-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
        }

        .skills-section h3 {
            font-size: 1em;
            margin-bottom: 12px;
        }

        .skill-tag {
            display: inline-block;
            padding: 6px 12px;
            border-radius: 20px;
            font-size: 0.85em;
            margin: 4px;
        }
        .skill-match   { background: rgba(46,204,113,0.2);  color: #2ecc71; }
        .skill-missing { background: rgba(231,76,60,0.2);   color: #e74c3c; }
        .skill-priority{ background: rgba(243,156,18,0.2);  color: #f39c12; }

        .loading {
            text-align: center;
            padding: 40px;
            display: none;
        }
        .spinner {
            width: 50px; height: 50px;
            border: 4px solid rgba(255,255,255,0.1);
            border-top-color: #00d2ff;
            border-radius: 50%;
            animation: spin 0.8s linear infinite;
            margin: 0 auto 16px;
        }
        @keyframes spin { to { transform: rotate(360deg); } }

        .filename-display {
            color: #00d2ff;
            text-align: center;
            margin-top: 8px;
            font-size: 0.9em;
        }
    </style>
</head>
<body>
<div class="container">

    <!-- Header -->
    <div class="header">
        <h1>🤖 AI Resume Screener</h1>
        <p>Upload your resume and paste a job description to get an instant match analysis</p>
    </div>

    <!-- Input Card -->
    <div class="card">
        <h2>📄 Step 1: Upload Your Resume</h2>
        <div class="file-upload" onclick="document.getElementById(\'pdfInput\').click()">
            <input type="file" id="pdfInput" accept=".pdf" onchange="handleFile(this)">
            <p>📁 <span>Click to upload</span> your resume PDF</p>
            <p style="font-size:0.8em; margin-top:8px;">Supported: PDF files only</p>
        </div>
        <div class="filename-display" id="filename"></div>

        <div class="or-divider">OR paste resume text</div>

        <textarea id="resumeText" rows="6"
                  placeholder="Paste your resume text here..."></textarea>
    </div>

    <div class="card">
        <h2>💼 Step 2: Paste Job Description</h2>
        <textarea id="jobText" rows="8"
                  placeholder="Paste the job description here..."></textarea>
    </div>

    <button class="btn" onclick="analyzeResume()">
        🔍 Analyze My Resume
    </button>

    <!-- Loading -->
    <div class="loading" id="loading">
        <div class="spinner"></div>
        <p>Analyzing your resume with AI...</p>
    </div>

    <!-- Results -->
    <div id="results">
        <div class="card" style="margin-top:24px;">
            <h2 style="text-align:center;">📊 Your Match Report</h2>

            <!-- Score Circle -->
            <div class="score-circle" id="scoreCircle">
                <span class="score-num" id="scoreNum">0%</span>
                <span class="score-label">Match Score</span>
            </div>

            <div class="status-badge" id="statusBadge"></div>
            <div class="advice" id="adviceText"></div>

            <!-- Score Breakdown -->
            <div class="score-breakdown">
                <div class="score-box">
                    <div class="val" id="tfidfScore">0%</div>
                    <div class="lbl">Content Similarity</div>
                </div>
                <div class="score-box">
                    <div class="val" id="skillsScore">0%</div>
                    <div class="lbl">Skills Match</div>
                </div>
            </div>

            <!-- Skills -->
            <div class="skills-grid">
                <div class="skills-section">
                    <h3>✅ Matching Skills (<span id="matchCount">0</span>)</h3>
                    <div id="matchingSkills"></div>
                </div>
                <div class="skills-section">
                    <h3>❌ Missing Skills (<span id="missingCount">0</span>)</h3>
                    <div id="missingSkills"></div>
                </div>
            </div>

            <!-- Priority -->
            <div style="margin-top:20px;">
                <h3 style="color:#f39c12; margin-bottom:10px;">
                    🎯 Top Priority Skills to Learn
                </h3>
                <div id="prioritySkills"></div>
            </div>
        </div>
    </div>

</div>

<script>
    let pdfFile = null;

    function handleFile(input) {
        pdfFile = input.files[0];
        document.getElementById("filename").textContent =
            "📎 " + pdfFile.name + " selected";
    }

    async function analyzeResume() {
        const jobText    = document.getElementById("jobText").value.trim();
        const resumeText = document.getElementById("resumeText").value.trim();

        if (!jobText) {
            alert("Please paste a job description!");
            return;
        }
        if (!resumeText && !pdfFile) {
            alert("Please upload a PDF or paste your resume text!");
            return;
        }

        // Show loading
        document.getElementById("loading").style.display = "block";
        document.getElementById("results").style.display = "none";

        const formData = new FormData();
        formData.append("job_description", jobText);
        if (pdfFile) formData.append("resume_pdf", pdfFile);
        else formData.append("resume_text", resumeText);

        try {
            const response = await fetch("/analyze", {
                method: "POST", body: formData
            });
            const data = await response.json();

            if (data.error) {
                alert("Error: " + data.error);
                return;
            }

            displayResults(data);

        } catch (err) {
            alert("Something went wrong. Please try again.");
        } finally {
            document.getElementById("loading").style.display = "none";
        }
    }

    function displayResults(data) {
        // Score circle
        document.getElementById("scoreNum").textContent  = data.final_score + "%";
        document.getElementById("scoreCircle").style.borderColor = data.color;
        document.getElementById("scoreCircle").style.color       = data.color;
        document.getElementById("statusBadge").textContent       = data.status;
        document.getElementById("statusBadge").style.color       = data.color;

        // Advice
        const adviceMap = {
            "STRONG MATCH"   : "Your resume is well aligned! Apply confidently. 🚀",
            "MODERATE MATCH" : "Good foundation! Add missing skills to strengthen your profile. 💪",
            "PARTIAL MATCH"  : "You have some relevant skills. Focus on the missing ones. 📚",
            "WEAK MATCH"     : "Significant gaps found. Consider upskilling before applying. 🎯"
        };
        document.getElementById("adviceText").textContent = adviceMap[data.status] || "";

        // Scores
        document.getElementById("tfidfScore").textContent  = data.tfidf_score + "%";
        document.getElementById("skillsScore").textContent = data.skills_score + "%";
        document.getElementById("matchCount").textContent  = data.matched_count;
        document.getElementById("missingCount").textContent= data.missing_count;

        // Matching skills
        document.getElementById("matchingSkills").innerHTML =
            data.matching_skills.map(s =>
                `<span class="skill-tag skill-match">${s}</span>`
            ).join("") || "<p style=\'color:#aaa\'>No matching skills found</p>";

        // Missing skills
        document.getElementById("missingSkills").innerHTML =
            data.missing_skills.map(s =>
                `<span class="skill-tag skill-missing">${s}</span>`
            ).join("") || "<p style=\'color:#aaa\'>No missing skills!</p>";

        // Priority skills
        document.getElementById("prioritySkills").innerHTML =
            data.top_3_missing.map(s =>
                `<span class="skill-tag skill-priority">🎯 ${s}</span>`
            ).join("") || "<p style=\'color:#aaa\'>You have all required skills!</p>";

        // Show results
        document.getElementById("results").style.display = "block";
        document.getElementById("results").scrollIntoView({ behavior: "smooth" });
    }
</script>
</body>
</html>
'''

# Save index.html
with open('/content/drive/MyDrive/CodTech_Internship/resume_screener/templates/index.html', 'w') as f:
    f.write(html_code)

print("✅ index.html saved to Google Drive!")

✅ index.html saved to Google Drive!


In [ ]:
# Install pyngrok to expose Flask app publicly
!pip install pyngrok -q

from pyngrok import ngrok
import threading
import os

# Copy app files to Colab local directory
os.makedirs('/content/resume_screener/templates', exist_ok=True)
os.makedirs('/content/resume_screener/uploads',   exist_ok=True)

# Copy files from Drive to Colab
import shutil
shutil.copy(
    '/content/drive/MyDrive/CodTech_Internship/resume_screener/app.py',
    '/content/resume_screener/app.py'
)
shutil.copy(
    '/content/drive/MyDrive/CodTech_Internship/resume_screener/templates/index.html',
    '/content/resume_screener/templates/index.html'
)

print("✅ Files copied!")

✅ Files copied!


In [ ]:
import subprocess
import time

# Start Flask app in background
os.chdir('/content/resume_screener')

flask_process = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for Flask to start
time.sleep(3)

# Create public URL using ngrok
ngrok.set_auth_token("3EEKaKt5cmuWq5UrCbfQdQKEQOC_2qCjNFr1xamYfuj7is4KN") # Leave empty for now
public_url = ngrok.connect(5000)

print("=" * 50)
print("   🚀 AI RESUME SCREENER IS LIVE!")
print("=" * 50)
print()
print(f"   🌐 Open this URL in your browser:")
print(f"   {public_url}")
print()
print("   Share this link with anyone!")
print("   Works on mobile too! 📱")
print("=" * 50)

   🚀 AI RESUME SCREENER IS LIVE!

   🌐 Open this URL in your browser:
   NgrokTunnel: "https://awkward-expiring-gnarly.ngrok-free.dev" -> "http://localhost:5000"

   Share this link with anyone!
   Works on mobile too! 📱


In [ ]:
fixed_app = '''
import os
import PyPDF2
import re
import spacy
from flask import Flask, request, render_template, jsonify
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from werkzeug.utils import secure_filename

app = Flask(__name__)
app.config["UPLOAD_FOLDER"] = "/tmp/uploads"
app.config["MAX_CONTENT_LENGTH"] = 16 * 1024 * 1024

os.makedirs("/tmp/uploads", exist_ok=True)

nlp = spacy.load("en_core_web_sm")

SKILLS_DATABASE = [
    "python", "java", "javascript", "sql", "r", "scala", "c++",
    "machine learning", "deep learning", "neural network", "nlp",
    "computer vision", "tensorflow", "pytorch", "keras", "scikit-learn",
    "pandas", "numpy", "matplotlib", "seaborn", "tableau", "power bi",
    "data analysis", "data visualization", "statistics",
    "flask", "fastapi", "django", "docker", "kubernetes", "aws",
    "azure", "gcp", "git", "github", "rest api",
    "mysql", "postgresql", "mongodb", "sqlite", "redis",
    "excel", "spark", "hadoop", "airflow", "etl pipeline",
    "random forest", "xgboost", "linear regression", "classification"
]

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, "rb") as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted
    except Exception as e:
        print(f"PDF error: {e}")
    return text

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    doc = nlp(text)
    words = [t.lemma_ for t in doc
             if not t.is_stop and not t.is_punct and len(t.text) > 2]
    return " ".join(words)

def extract_skills(text):
    text_lower = text.lower()
    return [s for s in SKILLS_DATABASE if s in text_lower]

def screen_resume(resume_text, job_description):
    cleaned_resume = preprocess_text(resume_text)
    cleaned_job    = preprocess_text(job_description)
    vectorizer     = TfidfVectorizer()
    vectors        = vectorizer.fit_transform([cleaned_resume, cleaned_job])
    tfidf_score    = round(cosine_similarity(vectors[0], vectors[1])[0][0] * 100, 2)

    resume_skills  = extract_skills(resume_text)
    job_skills     = extract_skills(job_description)
    matching       = [s for s in job_skills if s in resume_skills]
    missing        = [s for s in job_skills if s not in resume_skills]
    skills_score   = round((len(matching) / len(job_skills) * 100)
                            if job_skills else 0, 2)
    final_score    = round((tfidf_score * 0.4) + (skills_score * 0.6), 2)

    if final_score >= 75:
        status = "STRONG MATCH";    color = "#2ecc71"
    elif final_score >= 50:
        status = "MODERATE MATCH";  color = "#f39c12"
    elif final_score >= 30:
        status = "PARTIAL MATCH";   color = "#e67e22"
    else:
        status = "WEAK MATCH";      color = "#e74c3c"

    return {
        "final_score"     : final_score,
        "tfidf_score"     : tfidf_score,
        "skills_score"    : skills_score,
        "status"          : status,
        "color"           : color,
        "matching_skills" : matching,
        "missing_skills"  : missing,
        "top_3_missing"   : missing[:3],
        "matched_count"   : len(matching),
        "missing_count"   : len(missing),
        "total_skills"    : len(job_skills)
    }

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/analyze", methods=["POST"])
def analyze():
    try:
        job_description = request.form.get("job_description", "").strip()
        resume_text     = request.form.get("resume_text", "").strip()

        # Try PDF first
        if "resume_pdf" in request.files:
            file = request.files["resume_pdf"]
            if file and file.filename.endswith(".pdf"):
                filename = secure_filename(file.filename)
                filepath = os.path.join("/tmp/uploads", filename)
                file.save(filepath)
                pdf_text = extract_text_from_pdf(filepath)
                os.remove(filepath)
                if pdf_text.strip():
                    resume_text = pdf_text

        if not resume_text:
            return jsonify({"error": "Could not read resume. Please paste text instead."})
        if not job_description:
            return jsonify({"error": "Please provide a job description."})

        result = screen_resume(resume_text, job_description)
        return jsonify(result)

    except Exception as e:
        return jsonify({"error": str(e)})

if __name__ == "__main__":
    app.run(debug=False, port=5000)
'''

# Save fixed app.py
with open('/content/resume_screener/app.py', 'w') as f:
    f.write(fixed_app)

print("✅ Fixed app.py saved!")

✅ Fixed app.py saved!


In [ ]:
import subprocess, time
from pyngrok import ngrok

# Kill old process
subprocess.run(['pkill', '-f', 'app.py'], capture_output=True)
ngrok.kill()
time.sleep(2)

# Restart
os.chdir('/content/resume_screener')
flask_process = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(3)

ngrok.set_auth_token("3EEKaKt5cmuWq5UrCbfQdQKEQOC_2qCjNFr1xamYfuj7is4KN")
public_url = ngrok.connect(5000)

print("🚀 App restarted!")
print(f"🌐 New URL: {public_url}")

🚀 App restarted!
🌐 New URL: NgrokTunnel: "https://awkward-expiring-gnarly.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
# Install OCR library
!apt-get install -y tesseract-ocr
!pip install pytesseract pdf2image Pillow -q

print("✅ OCR libraries installed!")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ OCR libraries installed!


In [ ]:
fixed_app_v2 = '''
import os
import re
import spacy
import PyPDF2
import pytesseract
from PIL import Image
from pdf2image import convert_from_path
from flask import Flask, request, render_template, jsonify
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from werkzeug.utils import secure_filename

app = Flask(__name__)
os.makedirs("/tmp/uploads", exist_ok=True)

nlp = spacy.load("en_core_web_sm")

SKILLS_DATABASE = [
    "python", "java", "javascript", "sql", "r", "scala", "c++",
    "machine learning", "deep learning", "neural network", "nlp",
    "computer vision", "tensorflow", "pytorch", "keras", "scikit-learn",
    "pandas", "numpy", "matplotlib", "seaborn", "tableau", "power bi",
    "data analysis", "data visualization", "statistics",
    "flask", "fastapi", "django", "docker", "kubernetes", "aws",
    "azure", "gcp", "git", "github", "rest api",
    "mysql", "postgresql", "mongodb", "sqlite", "redis",
    "excel", "spark", "hadoop", "airflow", "etl pipeline",
    "random forest", "xgboost", "linear regression", "classification"
]

def extract_text_from_pdf(pdf_path):
    """
    First tries PyPDF2 (fast, for text-based PDFs)
    If that fails, uses OCR (for image-based/scanned PDFs)
    """
    text = ""

    # Method 1: PyPDF2 (works for text-based PDFs)
    try:
        with open(pdf_path, "rb") as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted
    except:
        pass

    # Method 2: OCR (works for image-based/scanned PDFs)
    if not text.strip():
        try:
            print("PyPDF2 failed — trying OCR...")
            images = convert_from_path(pdf_path, dpi=200)
            for image in images:
                ocr_text = pytesseract.image_to_string(image)
                text += ocr_text
            print("OCR successful!")
        except Exception as e:
            print(f"OCR error: {e}")

    return text.strip()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    doc = nlp(text)
    words = [t.lemma_ for t in doc
             if not t.is_stop and not t.is_punct and len(t.text) > 2]
    return " ".join(words)

def extract_skills(text):
    text_lower = text.lower()
    return [s for s in SKILLS_DATABASE if s in text_lower]

def screen_resume(resume_text, job_description):
    cleaned_resume = preprocess_text(resume_text)
    cleaned_job    = preprocess_text(job_description)
    vectorizer     = TfidfVectorizer()
    vectors        = vectorizer.fit_transform([cleaned_resume, cleaned_job])
    tfidf_score    = round(cosine_similarity(vectors[0], vectors[1])[0][0] * 100, 2)

    resume_skills  = extract_skills(resume_text)
    job_skills     = extract_skills(job_description)
    matching       = [s for s in job_skills if s in resume_skills]
    missing        = [s for s in job_skills if s not in resume_skills]
    skills_score   = round((len(matching) / len(job_skills) * 100)
                            if job_skills else 0, 2)
    final_score    = round((tfidf_score * 0.4) + (skills_score * 0.6), 2)

    if final_score >= 75:
        status = "STRONG MATCH";   color = "#2ecc71"
    elif final_score >= 50:
        status = "MODERATE MATCH"; color = "#f39c12"
    elif final_score >= 30:
        status = "PARTIAL MATCH";  color = "#e67e22"
    else:
        status = "WEAK MATCH";     color = "#e74c3c"

    return {
        "final_score"     : final_score,
        "tfidf_score"     : tfidf_score,
        "skills_score"    : skills_score,
        "status"          : status,
        "color"           : color,
        "matching_skills" : matching,
        "missing_skills"  : missing,
        "top_3_missing"   : missing[:3],
        "matched_count"   : len(matching),
        "missing_count"   : len(missing),
        "total_skills"    : len(job_skills)
    }

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/analyze", methods=["POST"])
def analyze():
    try:
        job_description = request.form.get("job_description", "").strip()
        resume_text     = request.form.get("resume_text", "").strip()

        # Try PDF first
        if "resume_pdf" in request.files:
            file = request.files["resume_pdf"]
            if file and file.filename.endswith(".pdf"):
                filename = secure_filename(file.filename)
                filepath = os.path.join("/tmp/uploads", filename)
                file.save(filepath)
                pdf_text = extract_text_from_pdf(filepath)
                os.remove(filepath)
                if pdf_text.strip():
                    resume_text = pdf_text

        if not resume_text:
            return jsonify({"error": "Could not read resume. Please paste text instead."})
        if not job_description:
            return jsonify({"error": "Please provide a job description."})

        result = screen_resume(resume_text, job_description)
        return jsonify(result)

    except Exception as e:
        return jsonify({"error": str(e)})

if __name__ == "__main__":
    app.run(debug=False, port=5000)
'''

# Save updated app.py
with open('/content/resume_screener/app.py', 'w') as f:
    f.write(fixed_app_v2)

print("✅ Updated app.py with OCR support saved!")

✅ Updated app.py with OCR support saved!


In [ ]:
import subprocess, time, os
from pyngrok import ngrok

# Kill old processes
subprocess.run(['pkill', '-f', 'app.py'], capture_output=True)
ngrok.kill()
time.sleep(2)

# Restart Flask
os.chdir('/content/resume_screener')
flask_process = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(4)

# New public URL
ngrok.set_auth_token("3EEKaKt5cmuWq5UrCbfQdQKEQOC_2qCjNFr1xamYfuj7is4KN")
public_url = ngrok.connect(5000)

print("🚀 App restarted with OCR support!")
print(f"🌐 URL: {public_url}")

🚀 App restarted with OCR support!
🌐 URL: NgrokTunnel: "https://awkward-expiring-gnarly.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
# Upload and test the PDF directly in Colab
from google.colab import files
uploaded = files.upload()

Saving Josey_John_Joseph_Resume_Minimal_v2.pdf to Josey_John_Joseph_Resume_Minimal_v2.pdf


In [ ]:
import PyPDF2
from pdf2image import convert_from_path
import pytesseract

# Get the uploaded filename
pdf_name = list(uploaded.keys())[0]

# Test Method 1: PyPDF2
print("Testing PyPDF2...")
with open(pdf_name, 'rb') as f:
    reader = PyPDF2.PdfReader(f)
    text1 = ""
    for page in reader.pages:
        text1 += page.extract_text() or ""
print(f"PyPDF2 result: '{text1[:200]}'")

# Test Method 2: OCR
print("\nTesting OCR...")
images = convert_from_path(pdf_name, dpi=150)
text2 = ""
for img in images:
    text2 += pytesseract.image_to_string(img)
print(f"OCR result: '{text2[:200]}'")

Testing PyPDF2...
PyPDF2 result: 'Josey John Joseph
+91 9495496295 joseyjjoseph023@gmail.com Bengaluru, India
P R O F I L E
Aspiring Data Science and Full-Stack professional with hands-on experience in ETL pipeline development, machin'

Testing OCR...


PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?

In [ ]:
fixed_app_v3 = '''
import os
import re
import spacy
import PyPDF2
import io
from flask import Flask, request, render_template, jsonify
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from werkzeug.utils import secure_filename

app = Flask(__name__)
nlp = spacy.load("en_core_web_sm")

SKILLS_DATABASE = [
    "python", "java", "javascript", "sql", "r", "scala", "c++",
    "machine learning", "deep learning", "neural network", "nlp",
    "computer vision", "tensorflow", "pytorch", "keras", "scikit-learn",
    "pandas", "numpy", "matplotlib", "seaborn", "tableau", "power bi",
    "data analysis", "data visualization", "statistics",
    "flask", "fastapi", "django", "docker", "kubernetes", "aws",
    "azure", "gcp", "git", "github", "rest api",
    "mysql", "postgresql", "mongodb", "sqlite", "redis",
    "excel", "spark", "hadoop", "airflow", "etl pipeline",
    "random forest", "xgboost", "linear regression", "classification"
]

def extract_text_from_pdf_bytes(pdf_bytes):
    """Read PDF directly from bytes — no file saving needed!"""
    text = ""
    try:
        reader = PyPDF2.PdfReader(io.BytesIO(pdf_bytes))
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted
    except Exception as e:
        print(f"PDF error: {e}")
    return text.strip()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    doc = nlp(text)
    words = [t.lemma_ for t in doc
             if not t.is_stop and not t.is_punct and len(t.text) > 2]
    return " ".join(words)

def extract_skills(text):
    return [s for s in SKILLS_DATABASE if s in text.lower()]

def screen_resume(resume_text, job_description):
    cleaned_resume = preprocess_text(resume_text)
    cleaned_job    = preprocess_text(job_description)
    vectorizer     = TfidfVectorizer()
    vectors        = vectorizer.fit_transform([cleaned_resume, cleaned_job])
    tfidf_score    = round(cosine_similarity(vectors[0], vectors[1])[0][0] * 100, 2)

    resume_skills = extract_skills(resume_text)
    job_skills    = extract_skills(job_description)
    matching      = [s for s in job_skills if s in resume_skills]
    missing       = [s for s in job_skills if s not in resume_skills]
    skills_score  = round((len(matching) / len(job_skills) * 100)
                           if job_skills else 0, 2)
    final_score   = round((tfidf_score * 0.4) + (skills_score * 0.6), 2)

    if final_score >= 75:
        status = "STRONG MATCH";   color = "#2ecc71"
    elif final_score >= 50:
        status = "MODERATE MATCH"; color = "#f39c12"
    elif final_score >= 30:
        status = "PARTIAL MATCH";  color = "#e67e22"
    else:
        status = "WEAK MATCH";     color = "#e74c3c"

    return {
        "final_score"     : final_score,
        "tfidf_score"     : tfidf_score,
        "skills_score"    : skills_score,
        "status"          : status,
        "color"           : color,
        "matching_skills" : matching,
        "missing_skills"  : missing,
        "top_3_missing"   : missing[:3],
        "matched_count"   : len(matching),
        "missing_count"   : len(missing),
        "total_skills"    : len(job_skills)
    }

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/analyze", methods=["POST"])
def analyze():
    try:
        job_description = request.form.get("job_description", "").strip()
        resume_text     = request.form.get("resume_text", "").strip()

        # Read PDF directly from bytes
        if "resume_pdf" in request.files:
            file = request.files["resume_pdf"]
            if file and file.filename.endswith(".pdf"):
                pdf_bytes = file.read()
                pdf_text  = extract_text_from_pdf_bytes(pdf_bytes)
                if pdf_text.strip():
                    resume_text = pdf_text

        if not resume_text:
            return jsonify({"error": "Could not read resume. Please paste text instead."})
        if not job_description:
            return jsonify({"error": "Please provide a job description."})

        result = screen_resume(resume_text, job_description)
        return jsonify(result)

    except Exception as e:
        return jsonify({"error": str(e)})

if __name__ == "__main__":
    app.run(debug=False, port=5000)
'''

with open('/content/resume_screener/app.py', 'w') as f:
    f.write(fixed_app_v3)

print("✅ Fixed app.py v3 saved!")


✅ Fixed app.py v3 saved!


In [ ]:
import subprocess, time, os
from pyngrok import ngrok

subprocess.run(['pkill', '-f', 'app.py'], capture_output=True)
ngrok.kill()
time.sleep(2)

os.chdir('/content/resume_screener')
flask_process = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(4)

ngrok.set_auth_token("3EEKaKt5cmuWq5UrCbfQdQKEQOC_2qCjNFr1xamYfuj7is4KN")
public_url = ngrok.connect(5000)

print("🚀 App restarted!")
print(f"🌐 URL: {public_url}")

🚀 App restarted!
🌐 URL: NgrokTunnel: "https://awkward-expiring-gnarly.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
import subprocess, time, os
from pyngrok import ngrok

subprocess.run(['pkill', '-f', 'app.py'], capture_output=True)
ngrok.kill()
time.sleep(2)

# Copy new HTML from Drive
import shutil
shutil.copy(
    '/content/drive/MyDrive/CodTech_Internship/resume_screener/templates/index.html',
    '/content/resume_screener/templates/index.html'
)

os.chdir('/content/resume_screener')
flask_process = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(4)

ngrok.set_auth_token("3EEKaKt5cmuWq5UrCbfQdQKEQOC_2qCjNFr1xamYfuj7is4KN")
public_url = ngrok.connect(5000)

print("🚀 App live with new design!")
print(f"🌐 {public_url}")

🚀 App live with new design!
🌐 NgrokTunnel: "https://awkward-expiring-gnarly.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
import subprocess, time, os
from pyngrok import ngrok

subprocess.run(['pkill', '-f', 'app.py'], capture_output=True)
ngrok.kill()
time.sleep(2)

# Copy new HTML from Drive
import shutil
shutil.copy(
    '/content/drive/MyDrive/CodTech_Internship/resume_screener/templates/index.html',
    '/content/resume_screener/templates/index.html'
)

os.chdir('/content/resume_screener')
flask_process = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(4)

ngrok.set_auth_token("3EEKaKt5cmuWq5UrCbfQdQKEQOC_2qCjNFr1xamYfuj7is4KN")
public_url = ngrok.connect(5000)

print("🚀 App live with new design!")
print(f"🌐 {public_url}")

🚀 App live with new design!
🌐 NgrokTunnel: "https://awkward-expiring-gnarly.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
import subprocess, time, os
from pyngrok import ngrok

subprocess.run(['pkill', '-f', 'app.py'], capture_output=True)
ngrok.kill()
time.sleep(2)

# Copy new HTML from Drive to Colab
import shutil
shutil.copy(
    '/content/drive/MyDrive/CodTech_Internship/resume_screener/templates/index.html',
    '/content/resume_screener/templates/index.html'
)

os.chdir('/content/resume_screener')
flask_process = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(4)

ngrok.set_auth_token("3EEKaKt5cmuWq5UrCbfQdQKEQOC_2qCjNFr1xamYfuj7is4KN")
public_url = ngrok.connect(5000)

print("🚀 App live with new design!")
print(f"🌐 {public_url}")

🚀 App live with new design!
🌐 NgrokTunnel: "https://awkward-expiring-gnarly.ngrok-free.dev" -> "http://localhost:5000"
